In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted!")


Mounted at /content/drive
Drive mounted!


In [ ]:
!pip install transformers torch datasets scikit-learn -q
print("Libraries installed!")

Libraries installed!


In [ ]:
from datasets import load_dataset

print("Loading GoEmotions dataset (may take 1-2 mins)...")
dataset = load_dataset("google-research-datasets/go_emotions", "simplified")

label_names = dataset['train'].features['labels'].feature.names

print("✅ Loaded!")
print(f"Train samples : {len(dataset['train'])}")
print(f"Val samples   : {len(dataset['validation'])}")
print(f"Test samples  : {len(dataset['test'])}")
print(f"\nTotal emotions: {len(label_names)}")
print("\nAll emotion labels:")
for i, name in enumerate(label_names):
    print(f"  {i:2d} → {name}")

Loading GoEmotions dataset (may take 1-2 mins)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

✅ Loaded!
Train samples : 43410
Val samples   : 5426
Test samples  : 5427

Total emotions: 28

All emotion labels:
   0 → admiration
   1 → amusement
   2 → anger
   3 → annoyance
   4 → approval
   5 → caring
   6 → confusion
   7 → curiosity
   8 → desire
   9 → disappointment
  10 → disapproval
  11 → disgust
  12 → embarrassment
  13 → excitement
  14 → fear
  15 → gratitude
  16 → grief
  17 → joy
  18 → love
  19 → nervousness
  20 → optimism
  21 → pride
  22 → realization
  23 → relief
  24 → remorse
  25 → sadness
  26 → surprise
  27 → neutral


In [ ]:
import pandas as pd

EMOTION_MAP = {
    # ATTENTIVE
    'admiration'    : 'Attentive',
    'curiosity'     : 'Attentive',
    'approval'      : 'Attentive',
    'excitement'    : 'Attentive',
    'joy'           : 'Attentive',
    'desire'        : 'Attentive',
    'optimism'      : 'Attentive',
    'amusement'     : 'Attentive',
    'realization'   : 'Attentive',
    'pride'         : 'Attentive',
    'love'          : 'Attentive',

    # BORED
    'neutral'       : 'Bored',
    'relief'        : 'Bored',
    'caring'        : 'Bored',
    'surprise'      : 'Bored',

    # CONFUSED
    'confusion'     : 'Confused',
    'nervousness'   : 'Confused',
    'fear'          : 'Confused',
    'embarrassment' : 'Confused',
    'disapproval'   : 'Confused',

    # FRUSTRATED
    'frustration'   : 'Frustrated',
    'anger'         : 'Frustrated',
    'annoyance'     : 'Frustrated',
    'disappointment': 'Frustrated',
    'sadness'       : 'Frustrated',
    'disgust'       : 'Frustrated',
    'grief'         : 'Frustrated',
    'remorse'       : 'Frustrated',
}

def convert_to_attention(example):
    if len(example['labels']) == 0:
        return None
    emotion_name = label_names[example['labels'][0]]
    return EMOTION_MAP.get(emotion_name, None)

print("Converting labels...")
rows = []
for split in ['train', 'validation', 'test']:
    for example in dataset[split]:
        attention = convert_to_attention(example)
        if attention is not None:
            rows.append({
                'text' : example['text'],
                'label': attention,
                'split': split
            })

df_nlp = pd.DataFrame(rows)
print(f"✅ Total samples : {len(df_nlp)}")
print("\nLabel distribution:")
print(df_nlp['label'].value_counts())

Converting labels...
✅ Total samples : 51582

Label distribution:
label
Attentive     20264
Bored         18288
Frustrated     8274
Confused       4756
Name: count, dtype: int64


In [ ]:
df_nlp.to_csv('/content/drive/MyDrive/DAiSEE/nlp_dataset.csv', index=False)
print("✅ NLP dataset saved to Drive!")

✅ NLP dataset saved to Drive!


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

df_nlp = pd.read_csv('/content/drive/MyDrive/DAiSEE/nlp_dataset.csv')

# Encode labels
le_nlp = LabelEncoder()
df_nlp['label_id'] = le_nlp.fit_transform(df_nlp['label'])

print("Label mapping:")
for i, cls in enumerate(le_nlp.classes_):
    print(f"  {i} → {cls}")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df_nlp['text'].tolist(),
    df_nlp['label_id'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_nlp['label_id']
)

print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

Label mapping:
  0 → Attentive
  1 → Bored
  2 → Confused
  3 → Frustrated

Train: 41265 | Test: 10317


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class AttentionTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = AttentionTextDataset(X_train, y_train, tokenizer)
test_dataset  = AttentionTextDataset(X_test,  y_test,  tokenizer)

train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=32)

print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Test batches : {len(test_loader)}")

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

✅ Train batches: 1290
✅ Test batches : 323


In [ ]:
from transformers import DistilBertForSequenceClassification

model_nlp = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=4
)
model_nlp.to(device)
print("✅ DistilBERT loaded and ready!")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ DistilBERT loaded and ready!


In [ ]:
# ✅ Stop current training and run this instead

import pandas as pd

# Load existing GoEmotions mapped data
df_go = pd.read_csv('/content/drive/MyDrive/DAiSEE/nlp_dataset.csv')

# ✅ Add high quality custom student responses
custom_data = [
    # ATTENTIVE
    ("I understand this concept and can explain it back", "Attentive"),
    ("This is really interesting, can you explain more?", "Attentive"),
    ("I see the connection between this and the previous topic", "Attentive"),
    ("The answer is because of how the algorithm optimizes", "Attentive"),
    ("I followed every step clearly", "Attentive"),
    ("Can we do a practice problem on this?", "Attentive"),
    ("I want to understand this better", "Attentive"),
    ("That makes complete sense to me now", "Attentive"),
    ("I was paying attention and I have a question", "Attentive"),
    ("This topic connects to what we studied before", "Attentive"),
    ("I understand the formula and how to apply it", "Attentive"),
    ("Can you give another example please", "Attentive"),
    ("I get it now the logic is very clear", "Attentive"),
    ("I am following along carefully", "Attentive"),
    ("This is a great explanation thank you", "Attentive"),
    ("I want to learn more about this topic", "Attentive"),
    ("The diagram makes it very clear", "Attentive"),
    ("I noticed something interesting about this pattern", "Attentive"),
    ("I think I can solve this problem now", "Attentive"),
    ("Could you explain the third step in more detail", "Attentive"),

    # BORED
    ("ok", "Bored"),
    ("yeah", "Bored"),
    ("sure whatever", "Bored"),
    ("how much longer is this class", "Bored"),
    ("can we take a break", "Bored"),
    ("I already know this stuff", "Bored"),
    ("this is not interesting at all", "Bored"),
    ("ok got it", "Bored"),
    ("can we skip this part", "Bored"),
    ("this is so repetitive", "Bored"),
    ("I don't see why we need to learn this", "Bored"),
    ("yeah yeah I know", "Bored"),
    ("fine ok", "Bored"),
    ("this is boring", "Bored"),
    ("I wish class was over", "Bored"),
    ("hmm ok sure", "Bored"),
    ("whatever you say", "Bored"),
    ("I guess so", "Bored"),
    ("can we do something else", "Bored"),
    ("this is taking too long", "Bored"),

    # CONFUSED
    ("I don't understand what you just explained", "Confused"),
    ("wait what does that mean", "Confused"),
    ("I am completely lost right now", "Confused"),
    ("can you repeat that please", "Confused"),
    ("this makes no sense to me", "Confused"),
    ("I thought it was the opposite", "Confused"),
    ("why does that happen I don't get it", "Confused"),
    ("can you start from the beginning again", "Confused"),
    ("I don't see how these two things connect", "Confused"),
    ("which formula are we supposed to use", "Confused"),
    ("I am confused about the second step", "Confused"),
    ("nothing is making sense right now", "Confused"),
    ("wait I missed something can you repeat", "Confused"),
    ("I don't follow the logic here", "Confused"),
    ("can you explain that differently", "Confused"),
    ("I keep getting confused between these two concepts", "Confused"),
    ("what is the difference between these", "Confused"),
    ("I have no idea what is happening", "Confused"),
    ("this is confusing me a lot", "Confused"),
    ("I thought I understood but now I am lost", "Confused"),

    # FRUSTRATED
    ("this is too hard I give up", "Frustrated"),
    ("I have been trying but I still don't get it", "Frustrated"),
    ("why is this so complicated", "Frustrated"),
    ("I studied this but still can't understand", "Frustrated"),
    ("this is really frustrating me", "Frustrated"),
    ("I keep getting the wrong answer", "Frustrated"),
    ("I don't think I will ever understand this", "Frustrated"),
    ("why does this have to be so hard", "Frustrated"),
    ("I am tired of trying and failing", "Frustrated"),
    ("nothing I do seems to work", "Frustrated"),
    ("I want to give up on this", "Frustrated"),
    ("every time I think I understand I get it wrong", "Frustrated"),
    ("this is way too difficult for me", "Frustrated"),
    ("I feel like I am the only one who doesn't get it", "Frustrated"),
    ("I have tried everything and still failing", "Frustrated"),
    ("this makes me feel stupid", "Frustrated"),
    ("I can't do this no matter how hard I try", "Frustrated"),
    ("this subject is impossible", "Frustrated"),
    ("I am so stressed about not understanding this", "Frustrated"),
    ("I give up I will never learn this", "Frustrated"),
]

df_custom = pd.DataFrame(custom_data, columns=['text', 'label'])

# ✅ Combine both datasets
df_combined = pd.concat([df_go, df_custom], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"GoEmotions samples : {len(df_go)}")
print(f"Custom samples     : {len(df_custom)}")
print(f"Combined total     : {len(df_combined)}")
print("\nLabel distribution:")
print(df_combined['label'].value_counts())

# Save
df_combined.to_csv('/content/drive/MyDrive/DAiSEE/nlp_dataset_v2.csv', index=False)
print("\n✅ Combined dataset saved!")

GoEmotions samples : 51582
Custom samples     : 80
Combined total     : 51662

Label distribution:
label
Attentive     20284
Bored         18308
Frustrated     8294
Confused       4776
Name: count, dtype: int64

✅ Combined dataset saved!


NEW APPROACH



In [ ]:
from datasets import load_dataset
import pandas as pd

# ✅ Dataset 1: dair-ai/emotion
print("Loading dair-ai/emotion...")
emotion_ds = load_dataset("dair-ai/emotion", "split")
print(f"✅ Loaded! Train: {len(emotion_ds['train'])} samples")
print("Sample:", emotion_ds['train'][0])
print("Labels:", emotion_ds['train'].features['label'].names)

Loading dair-ai/emotion...


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

✅ Loaded! Train: 16000 samples
Sample: {'text': 'i didnt feel humiliated', 'label': 0}
Labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [ ]:
# Check dair-ai/emotion labels and samples
print("Label names:", emotion_ds['train'].features['label'].names)
print(f"\nTrain: {len(emotion_ds['train'])}")
print(f"Val  : {len(emotion_ds['validation'])}")
print(f"Test : {len(emotion_ds['test'])}")
print("\nSample:")
print(emotion_ds['train'][0])

# Count per label
import pandas as pd
df_emotion = pd.DataFrame(emotion_ds['train'])
print("\nLabel distribution:")
print(df_emotion['label'].value_counts())

Label names: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

Train: 16000
Val  : 2000
Test : 2000

Sample:
{'text': 'i didnt feel humiliated', 'label': 0}

Label distribution:
label
1    5362
0    4666
3    2159
4    1937
2    1304
5     572
Name: count, dtype: int64


In [ ]:
import pandas as pd

# ✅ Map dair-ai/emotion to our 4 classes
label_names = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

EMOTION_MAP_DAIR = {
    'joy'     : 'Attentive',
    'love'    : 'Attentive',
    'surprise': 'Attentive',
    'fear'    : 'Confused',
    'sadness' : 'Frustrated',
    'anger'   : 'Frustrated',
    # Note: no 'Bored' in dair — GoEmotions covers that
}

rows = []
for split in ['train', 'validation', 'test']:
    for example in emotion_ds[split]:
        label_name = label_names[example['label']]
        attention  = EMOTION_MAP_DAIR.get(label_name, None)
        if attention:
            rows.append({
                'text' : example['text'],
                'label': attention
            })

df_dair = pd.DataFrame(rows)
print(f"dair-ai/emotion mapped: {len(df_dair)} samples")
print(df_dair['label'].value_counts())

# ✅ Combine with GoEmotions
df_go       = pd.read_csv('/content/drive/MyDrive/DAiSEE/nlp_dataset.csv')
df_go       = df_go[['text', 'label']]

df_combined = pd.concat([df_go, df_dair], ignore_index=True)
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Combined dataset: {len(df_combined)} samples")
print("\nFinal label distribution:")
print(df_combined['label'].value_counts())

# Save
df_combined.to_csv('/content/drive/MyDrive/DAiSEE/nlp_dataset_final.csv', index=False)
print("\n✅ Saved!")

dair-ai/emotion mapped: 20000 samples
label
Attentive     9121
Frustrated    8506
Confused      2373
Name: count, dtype: int64

✅ Combined dataset: 71582 samples

Final label distribution:
label
Attentive     29385
Bored         18288
Frustrated    16780
Confused       7129
Name: count, dtype: int64

✅ Saved!


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, classification_report
import torch

# Encode labels
le_nlp = LabelEncoder()
df_combined['label_id'] = le_nlp.fit_transform(df_combined['label'])

print("Label mapping:")
for i, cls in enumerate(le_nlp.classes_):
    print(f"  {i} → {cls}")

# Split
X_train, X_test, y_train, y_test = train_test_split(
    df_combined['text'].tolist(),
    df_combined['label_id'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_combined['label_id']
)

print(f"\nTrain: {len(X_train)} | Test: {len(X_test)}")

# Dataset class
class AttentionTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

train_dataset = AttentionTextDataset(X_train, y_train, tokenizer)
test_dataset  = AttentionTextDataset(X_test,  y_test,  tokenizer)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader   = DataLoader(test_dataset,  batch_size=32)

# Fresh model
model_nlp = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=4
)
model_nlp.to(device)

# Training
optimizer   = AdamW(model_nlp.parameters(), lr=2e-5, weight_decay=0.01)
EPOCHS      = 8
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = total_steps // 10,
    num_training_steps = total_steps
)

print(f"\nTraining on {len(df_combined)} samples for {EPOCHS} epochs...\n")
best_acc = 0

for epoch in range(EPOCHS):
    model_nlp.train()
    total_loss = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model_nlp(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_nlp.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # Evaluate
    model_nlp.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs        = model_nlp(input_ids=input_ids, attention_mask=attention_mask)
            preds          = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"✅ Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Accuracy: {acc*100:.2f}%")

    if acc > best_acc:
        best_acc = acc
        torch.save(model_nlp.state_dict(), '/content/best_nlp_final.pt')
        print(f"   💾 Best saved! ({acc*100:.2f}%)")

print(f"\n🏆 Best Accuracy: {best_acc*100:.2f}%")
print(classification_report(all_labels, all_preds, target_names=le_nlp.classes_))


Label mapping:
  0 → Attentive
  1 → Bored
  2 → Confused
  3 → Frustrated

Train: 57265 | Test: 14317


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training on 71582 samples for 8 epochs...



KeyboardInterrupt: 

In [ ]:
import os
import pickle
from transformers import DistilBertConfig, DistilBertForSequenceClassification

# ✅ Checkpoint path on Drive
checkpoint_dir = '/content/drive/MyDrive/DAiSEE/nlp_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

config = DistilBertConfig.from_pretrained(
    'distilbert-base-uncased',
    num_labels=4,
    dropout=0.3,
    attention_dropout=0.2
)

model_nlp = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    config=config
)
model_nlp.to(device)

optimizer   = AdamW(model_nlp.parameters(), lr=1e-5, weight_decay=0.05)
EPOCHS      = 10
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = total_steps // 5,
    num_training_steps = total_steps
)

# ✅ Resume logic — check if checkpoint exists
start_epoch = 0
best_acc    = 0
no_improve  = 0
patience    = 3

checkpoint_file = f'{checkpoint_dir}/latest_checkpoint.pt'
best_model_file = f'{checkpoint_dir}/best_model.pt'
meta_file       = f'{checkpoint_dir}/training_meta.pkl'

if os.path.exists(checkpoint_file) and os.path.exists(meta_file):
    print("✅ Found checkpoint! Resuming training...")
    model_nlp.load_state_dict(torch.load(checkpoint_file))

    with open(meta_file, 'rb') as f:
        meta = pickle.load(f)

    start_epoch = meta['epoch'] + 1
    best_acc    = meta['best_acc']
    no_improve  = meta['no_improve']
    print(f"   Resuming from epoch {start_epoch + 1}")
    print(f"   Best accuracy so far: {best_acc*100:.2f}%")
else:
    print("🆕 Starting fresh training...")

print(f"\nTraining epochs {start_epoch+1} to {EPOCHS}...\n")

for epoch in range(start_epoch, EPOCHS):
    model_nlp.train()
    total_loss = 0

    for i, batch in enumerate(train_loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model_nlp(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_nlp.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # Evaluate
    model_nlp.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)
            outputs        = model_nlp(input_ids=input_ids, attention_mask=attention_mask)
            preds          = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"✅ Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Accuracy: {acc*100:.2f}%")

    # ✅ Save checkpoint every epoch to Drive
    torch.save(model_nlp.state_dict(), checkpoint_file)
    with open(meta_file, 'wb') as f:
        pickle.dump({
            'epoch'     : epoch,
            'best_acc'  : best_acc,
            'no_improve': no_improve
        }, f)
    print(f"   💾 Checkpoint saved to Drive (epoch {epoch+1})")

    # Save best model
    if acc > best_acc:
        best_acc   = acc
        no_improve = 0
        torch.save(model_nlp.state_dict(), best_model_file)
        print(f"   🏆 Best model saved! ({acc*100:.2f}%)")
    else:
        no_improve += 1
        print(f"   ⚠️ No improvement ({no_improve}/{patience})")
        if no_improve >= patience:
            print(f"\n🛑 Early stopping at epoch {epoch+1}")
            break

# ✅ Final save to permanent location
print("\nSaving final model to Drive...")
model_nlp.load_state_dict(torch.load(best_model_file))
nlp_path = '/content/drive/MyDrive/DAiSEE/distilbert_nlp_final'
model_nlp.save_pretrained(nlp_path)
tokenizer.save_pretrained(nlp_path)

with open('/content/drive/MyDrive/DAiSEE/le_nlp_final.pkl', 'wb') as f:
    pickle.dump(le_nlp, f)

print(f"\n✅ All saved!")
print(f"   → {nlp_path}")
print(f"   → le_nlp_final.pkl")
print(f"\n🏆 Best Accuracy: {best_acc*100:.2f}%")
print(classification_report(all_labels, all_preds, target_names=le_nlp.classes_))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🆕 Starting fresh training...

Training epochs 1 to 10...

✅ Epoch 1/10 | Loss: 1.0735 | Accuracy: 67.56%
   💾 Checkpoint saved to Drive (epoch 1)
   🏆 Best model saved! (67.56%)
✅ Epoch 2/10 | Loss: 0.7533 | Accuracy: 72.03%
   💾 Checkpoint saved to Drive (epoch 2)
   🏆 Best model saved! (72.03%)
✅ Epoch 3/10 | Loss: 0.6491 | Accuracy: 72.93%
   💾 Checkpoint saved to Drive (epoch 3)
   🏆 Best model saved! (72.93%)
✅ Epoch 4/10 | Loss: 0.5980 | Accuracy: 74.93%
   💾 Checkpoint saved to Drive (epoch 4)
   🏆 Best model saved! (74.93%)
✅ Epoch 5/10 | Loss: 0.5623 | Accuracy: 74.84%
   💾 Checkpoint saved to Drive (epoch 5)
   ⚠️ No improvement (1/3)
✅ Epoch 6/10 | Loss: 0.5365 | Accuracy: 75.38%
   💾 Checkpoint saved to Drive (epoch 6)
   🏆 Best model saved! (75.38%)
✅ Epoch 7/10 | Loss: 0.5155 | Accuracy: 75.39%
   💾 Checkpoint saved to Drive (epoch 7)
   🏆 Best model saved! (75.39%)
✅ Epoch 8/10 | Loss: 0.5013 | Accuracy: 74.89%
   💾 Checkpoint saved to Drive (epoch 8)
   ⚠️ No improvemen

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ All saved!
   → /content/drive/MyDrive/DAiSEE/distilbert_nlp_final
   → le_nlp_final.pkl

🏆 Best Accuracy: 75.39%
              precision    recall  f1-score   support

   Attentive       0.80      0.85      0.82      5877
       Bored       0.69      0.54      0.61      3658
    Confused       0.60      0.62      0.61      1426
  Frustrated       0.76      0.83      0.79      3356

    accuracy                           0.75     14317
   macro avg       0.71      0.71      0.71     14317
weighted avg       0.74      0.75      0.74     14317

